In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# CITATION INFORMATION
# ============================================================
# If you find this code useful for your research, please cite:
#
# Muharrem BALCI, STATISTICAL RELIABILITY AND EXPLAINABILITY
# OF MODERN CONVNEXTV2 AND SWIN TRANSFORMER ARCHITECTURES IN THE
# CLASSIFICATION OF MULTIPLE RETINAL DISEASES BASED ON FUNDUS IMAGES,
# (Submitted for publication), 2026.
#
# GitHub: https://github.com/mblci/Retina-Diseases-DeepLearning-Benchmark
# ============================================================

# ============================================================
# ABLATION STUDY: TRAINING ON ORIGINAL DATASET (NO AUGMENTATION)
# ============================================================
# Description: This script trains the ConvNeXtV2 model using only
# the original fundus images to evaluate the impact of data
# augmentation by comparing results with the augmented pipeline.
#
# Content: Data splitting (85/10/5), training (AMP), early stopping,
# performance metrics (F1, Precision, Recall), Confusion Matrix,
# ROC/PR curves, and misclassification analysis.
#
# Dataset Source: [Eye Disease Image Dataset: https://data.mendeley.com/datasets/s9bfhswzjb/1]
# ============================================================

import os
import random
import shutil
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from timm import create_model
import torch.optim as optim
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

# -----------------------------
# 1- Device Configuration
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# -----------------------------
# 2- Data Paths and Splitting
# -----------------------------
# Using original dataset for ablation study
data_dir = "./dataset/original"
base_dir = "./dataset/original_split"
results_dir = "./results/ablation_study_original_metrics"
os.makedirs(results_dir, exist_ok=True)

if not os.path.exists(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(base_dir, split), exist_ok=True)

    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    for cls in classes:
        cls_path = os.path.join(data_dir, cls)
        images = [f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))]
        random.shuffle(images)

        n_total = len(images)
        n_train = int(0.85 * n_total)
        n_val = int(0.10 * n_total)
        n_test = n_total - n_train - n_val

        splits = {
            "train": images[:n_train],
            "val": images[n_train:n_train + n_val],
            "test": images[n_train + n_val:]
        }

        for split, imgs in splits.items():
            split_dir = os.path.join(base_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)
            for img in imgs:
                shutil.copy(os.path.join(cls_path, img),
                            os.path.join(split_dir, img))
    print(" Data split 85/10/5 completed.")
else:
    print(" Split data already exists, skipping creation.")

# -----------------------------
# 3- Transforms & Safe Loader
# -----------------------------
def pil_loader(path):
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')
    except Exception as e:
        print(f" Corrupted image skipped: {path} ({e})")
        return Image.new("RGB", (224, 224), (0, 0, 0))

IMG_SIZE = 224
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Minimal transforms for ablation study (No heavy augmentation)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), # Minimal standard augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_data = datasets.ImageFolder(root=os.path.join(base_dir, "train"), transform=train_transform, loader=pil_loader)
val_data = datasets.ImageFolder(root=os.path.join(base_dir, "val"), transform=val_test_transform, loader=pil_loader)
test_data = datasets.ImageFolder(root=os.path.join(base_dir, "test"), transform=val_test_transform, loader=pil_loader)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(train_data.classes)
print(f"Class count: {num_classes}, Class names: {train_data.classes}")

# -----------------------------
# 4- Model Selection (Ablation Focus: ConvNeXtV2)
# -----------------------------
model_list = {
    "EfficientNetV2_S": "tf_efficientnetv2_s.in21k_ft_in1k",
    "Swin_Tiny": "swin_tiny_patch4_window7_224",
    "ConvNeXtV2_Base": "convnextv2_base",
}

model_name = "ConvNeXtV2_Base"
model_timm = model_list[model_name]
model = create_model(model_timm, pretrained=True, num_classes=num_classes)
model.to(device)
print(f"Using Model: {model_name}")

# Model parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
with open(os.path.join(results_dir, f"model_info_{model_name}.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Trainable parameters: {total_params:,}\n")

# -----------------------------
# 5- Training Configuration
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# -----------------------------
# 6- Training Loop (AMP + Early Stopping)
# -----------------------------
epochs = 15
best_val_loss = float('inf')
patience = 10
counter = 0
scaler = GradScaler()

history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    print(f"\nEpoch [{epoch+1}/{epochs}]")
    model.train()
    train_loss = 0.0
    running_samples = 0

    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        running_samples += images.size(0)

    train_loss = train_loss / running_samples
    scheduler.step()

    # Validation
    model.eval()
    val_loss = 0.0
    val_samples = 0
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_samples += images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss / val_samples
    val_acc = correct / total
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), os.path.join(results_dir, f"best_{model_name}.pth"))
        print(" Best model saved!")
    else:
        counter += 1
        if counter >= patience:
            print(f" Early stopping triggered after {patience} epochs without improvement.")
            break

# -----------------------------
# 7- Post-Training Visualization
# -----------------------------
# Save History
history_csv = os.path.join(results_dir, f"history_{model_name}.csv")
pd.DataFrame(history).to_csv(history_csv, index=False)

# Plots
plt.figure(figsize=(10,5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title(f"Loss Curve (Ablation) - {model_name}")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig(os.path.join(results_dir, "loss_curve.png"))
plt.show()

# -----------------------------
# 8- Final Evaluation (Test Set)
# -----------------------------
model.load_state_dict(torch.load(os.path.join(results_dir, f"best_{model_name}.pth")))
model.eval()
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true, y_pred, y_prob = np.array(y_true), np.array(y_pred), np.array(y_prob)

# Save Metrics
metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "F1 (Macro)": f1_score(y_true, y_pred, average='macro'),
}
print("\n Ablation Test Metrics:", metrics)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=train_data.classes, yticklabels=train_data.classes)
plt.title("Confusion Matrix (Original Data Only)")
plt.savefig(os.path.join(results_dir, "confusion_matrix.png"))
plt.show()

print(f" Ablation study execution completed for {model_name}. Results saved to {results_dir}")